# 5.3 Claims retrieval: does a career get research that is about it?

"What research says" (4.1c) picks claims for each career in two steps:
1. **Retrieval:** BGE embeddings rank the eligible claims for the career; each career keeps its top 10 by two scores, *raw* cosine similarity and *centered* (raw minus the claim's average over all careers), plus every claim the relevance check already judged relevant for it (CHANGELOG 058).
2. **Relevance check:** an LLM answers which candidates are about the career; the app shows at most 3, highest centered score first.

**Ground truth:** `evaluation/claim_relevance_labels.csv`, 349 candidates for 40 careers from 22 occupation groups (each career's top 5 by raw ∪ top 5 by centered), judged 1 if the claim is about the career, its field or industry, or its main tasks. **Who judged:** Claude drafted the labels without seeing the scores, and you accepted them (CHANGELOG 031). Only these pooled candidates are judged, so recall here means *share of the known relevant claims found*.

**Later rounds** (`evaluation/claim_relevance_labels_<round>.csv`, ROADMAP 8.4) add, for the same careers, the pairs new claims brought into that pool or into what the app shows, judged the same way; earlier labels are kept. Round 2026_10: 87 pairs after the first monthly update (CHANGELOG 060).

**New claims (ROADMAP 8.4):** the labels judge the June 2026 candidates. Claims added since (Phase 8) reach the candidates and what the app shows without a label; section 2 counts how many, and when they pass about 1 in 5 of what is shown, a new labelling round is due.

Reads local files only: the labels, `data/duckdb/research_claims.duckdb` and `data/research/claim_relevance_checks.jsonl` (the saved LLM answers; untracked, so this notebook runs where the build ran). No API calls; the embedding model loads from the local cache for the nonsense check.

In [1]:
import _setup
_setup.setup()

In [2]:
import json
import os
import sys
import warnings
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

os.environ["HF_HUB_OFFLINE"] = "1"  # the embedding model comes from the local cache
os.environ["TQDM_DISABLE"] = "1"  # no progress bars in the saved output (their timings change every run)
warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm asking for Jupyter widgets
sys.path.insert(0, "scripts/research")
import check_claim_relevance as relevance  # noqa: E402
import match_claims_to_occupations as matcher  # noqa: E402

labels = pd.concat(pd.read_csv(path, dtype={"relevant": int})  # June 2026, then each later round (ROADMAP 8.4)
                   for path in sorted(Path("evaluation").glob("claim_relevance_labels*.csv")))
con = duckdb.connect(str(matcher.OUTPUT), read_only=True)
candidates = con.execute(
    "SELECT c.onetsoc_code, c.claim_id, c.score, c.centered, r.claim_text "
    "FROM claim_candidates c JOIN research_claims r USING (claim_id)").df()
con.close()
judged = candidates.merge(labels[["onetsoc_code", "claim_id", "relevant"]], on=["onetsoc_code", "claim_id"])
codes = sorted(labels.onetsoc_code.unique())
known = judged.groupby("onetsoc_code").relevant.sum()
print(f"judged: {len(judged)} candidates, {len(codes)} careers, {judged.relevant.sum()} relevant; "
      f"careers with at least one relevant candidate: {(known > 0).sum()} (the ceiling for any filter)")

judged: 422 candidates, 40 careers, 100 relevant; careers with at least one relevant candidate: 31 (the ceiling for any filter)


## 1. Ranking: raw vs. centered score

Per career, then averaged over careers. The top 5 by each score are judged (each labelling round adds what new claims brought in), so the metrics stop at 5. Claims added after the latest round have no label and are skipped: these metrics rank the judged candidates only.
- **Precision@k:** share of the first k that are relevant (all 40 careers).
- **Recall@5:** share of the career's known relevant claims that are in its top 5 (the careers that have any).
- **MRR@5:** 1 / rank of the first relevant claim, 0 if none in the top 5 (the careers that have any).

In [3]:
def ranking_metrics(column):
    rows = []
    for code, group in judged.groupby("onetsoc_code"):
        top = group.sort_values(column, ascending=False).relevant.to_numpy()[:5]
        first = np.flatnonzero(top)
        rows.append({"p@1": top[:1].mean(), "p@3": top[:3].mean(), "p@5": top.mean(),
                     "recall@5": top.sum() / known[code] if known[code] else np.nan,
                     "mrr@5": 1 / (first[0] + 1) if len(first) else (0.0 if known[code] else np.nan)})
    return pd.DataFrame(rows).mean().round(3)


pd.DataFrame({"raw": ranking_metrics("score"), "centered": ranking_metrics("centered")})

,raw,centered
p@1,0.425,0.550
p@3,0.333,0.367
p@5,0.290,0.305
recall@5,0.569,0.654
mrr@5,0.645,0.771


## 2. What a student would see

Each rule shows at most 3 claims per career. *Precision* is the share of shown claims that are relevant (by the labels; shown claims outside the judged pool are counted separately). The 95% interval is a bootstrap over careers, since 40 careers is a small sample.

Rules compared:
- **threshold only** (D16 option A): centered score ≥ a minimum;
- **LLM check** (D16 option C) with each model tried, then the app's rule (`select_shown`: judged relevant, centered ≥ 0, one per text, highest centered first). The app also leaves out claims that don't stand on their own (ROADMAP 4.2), which this comparison of relevance rules doesn't; the next cell counts what it really shows. Only gpt-5.4-mini, the app's model, checked the claims added after June 2026, so the other models never show them: compare models on precision, not on how many careers they reach.

In [4]:
answers = {}
for line in open(relevance.CHECKS, encoding="utf-8"):
    row = json.loads(line)
    for claim_id, value in row["answers"].items():
        answers[(row["model"], row["onetsoc_code"], claim_id)] = value
MODELS = ["gpt-4o-mini", "gpt-4.1-mini", "gpt-5.4-nano", "gpt-5.4-mini"]
label_of = {(row.onetsoc_code, row.claim_id): row.relevant for row in judged.itertuples()}
by_code = {code: group for code, group in candidates[candidates.onetsoc_code.isin(codes)].groupby("onetsoc_code")}
rng = np.random.default_rng(2026)


def shown_ids(code, rule):
    group = by_code[code]
    if rule[0] == "threshold":
        return list(group[group.centered >= rule[1]].sort_values("centered", ascending=False).claim_id[:3])
    relevant = {cid for cid in group.claim_id if answers.get((rule[1], code, cid))}
    return relevance.select_shown(list(zip(group.claim_id, group.claim_text, group.centered)), relevant)


def outcome(rule):
    per_code = {code: [label_of.get((code, cid)) for cid in shown_ids(code, rule)] for code in codes}
    hits = {code: [v for v in values if v is not None] for code, values in per_code.items()}
    flat = [v for values in hits.values() for v in values]
    boot = []
    for _ in range(2000):
        sample = [v for code in rng.choice(codes, len(codes)) for v in hits[code]]
        boot.append(np.mean(sample) if sample else np.nan)
    low, high = np.nanpercentile(boot, [2.5, 97.5])
    return {"shown (judged)": len(flat), "shown (not judged)": sum(v is None for vs in per_code.values() for v in vs),
            "precision": round(np.mean(flat), 3), "95% interval": f"{low:.2f}-{high:.2f}",
            "careers showing any": sum(bool(v) for v in per_code.values()),
            "careers with a relevant one": sum(any(v) for v in hits.values())}


rules = {f"threshold >= {t}": ("threshold", t) for t in (0.10, 0.12, 0.13, 0.14)}
rules |= {f"LLM check: {model}": ("llm", model) for model in MODELS}
pd.DataFrame({name: outcome(rule) for name, rule in rules.items()}).T

,shown (judged),shown (not judged),precision,95% interval,careers showing any,careers with a relevant one
threshold >= 0.1,72,0,0.431,0.27-0.59,30,18
threshold >= 0.12,43,0,0.512,0.29-0.72,20,11
threshold >= 0.13,30,0,0.633,0.37-0.83,15,10
threshold >= 0.14,23,0,0.652,0.39-0.90,10,8
LLM check: gpt-4o-mini,43,6,0.698,0.45-0.89,20,14
LLM check: gpt-4.1-mini,65,14,0.615,0.45-0.78,32,20
LLM check: gpt-5.4-nano,57,13,0.632,0.45-0.80,27,18
LLM check: gpt-5.4-mini,84,0,0.667,0.55-0.79,34,29


In [5]:
app = pd.read_csv(relevance.SHOWN, dtype=str)  # what the app shows: also without claims that don't stand on their own (4.2)
shown = [label_of.get((code, cid)) for code, cid in zip(app.onetsoc_code, app.claim_id) if code in codes]
unjudged = sum(v is None for v in shown)
print(f"what the app shows: {unjudged} of {len(shown)} claims shown for the 40 careers are outside the "
      f"judged pool ({unjudged / len(shown):.0%})" + ("; more than 1 in 5, so a new labelling round is due (ROADMAP 8.4)"
                                                 if unjudged / len(shown) > 0.2 else ""))

what the app shows: 5 of 80 claims shown for the 40 careers are outside the judged pool (6%)


## 3. The LLM's answers vs. the labels

On the judged pairs that are still candidates, each model on the pairs it answered: only gpt-5.4-mini, the app's model, checked the pairs from later rounds. Cohen's κ corrects agreement for chance, which matters here because most candidates are not relevant (an LLM saying "no" to everything would agree 79% of the time).

In [6]:
rows = {}
for model in MODELS:
    pairs = [(row.relevant, int(answers[(model, row.onetsoc_code, row.claim_id)])) for row in judged.itertuples()
             if (model, row.onetsoc_code, row.claim_id) in answers]
    truth, said = np.array(pairs).T
    rows[model] = {"agreement": (truth == said).mean(), "precision": truth[said == 1].mean(),
                   "recall": said[truth == 1].mean(), "kappa": cohen_kappa_score(truth, said), "said yes": said.sum(), "pairs": len(pairs)}
pd.DataFrame(rows).T.round(3)

,agreement,precision,recall,kappa,said yes,pairs
gpt-4o-mini,0.836,0.704,0.481,0.474,54.0,347.0
gpt-4.1-mini,0.836,0.631,0.671,0.543,84.0,347.0
gpt-5.4-nano,0.853,0.659,0.734,0.598,88.0,347.0
gpt-5.4-mini,0.884,0.726,0.820,0.693,113.0,422.0


## 4. Queries that should find nothing

In [7]:
vectors = np.load(matcher.OUTPUT.with_suffix(".claim_vectors.npy"))
claim_means = np.load(matcher.OUTPUT.with_suffix(".claim_means.npy"))
nonsense = ["asdf qwerty zxcv", "chocolate cake recipe with strawberries", "the history of the Roman empire"]
query_vectors = matcher.embed([matcher.BGE_QUERY_PREFIX + q for q in nonsense])
real_top = candidates.groupby("onetsoc_code")[["score", "centered"]].max()
pd.DataFrame(
    [{"query": q, "best raw": (v @ vectors.T).max(), "best centered": (v @ vectors.T - claim_means).max()}
     for q, v in zip(nonsense, query_vectors)]
    + [{"query": "real careers: lowest best score", "best raw": real_top.score.min(), "best centered": real_top.centered.min()},
       {"query": "real careers: median best score", "best raw": real_top.score.median(), "best centered": real_top.centered.median()}]
).round(3)

,query,best raw,best centered
0,asdf qwerty zxcv,0.545,0.056
1,chocolate cake recipe with strawberries,0.435,-0.026
2,the history of the Roman empire,0.572,0.061
3,real careers: lowest best score,0.565,0.032
4,real careers: median best score,0.687,0.122


## Findings

1. **Centering helps.** The centered score beats the raw one on every ranking metric: generic claims ("AI can perform some of the subtasks in an occupation") no longer win for every career.
2. **Similarity alone is not enough.** The best threshold shows claims for few careers, and about a third of what it shows is off-topic.
3. **An LLM relevance check fixes most of that, but the model matters.** gpt-5.4-mini reaches the highest agreement and κ, and gives a relevant claim to the most careers, close to the ceiling. gpt-4o-mini, the app's default model, says yes too rarely and misses half the relevant claims. The intervals are wide: 40 careers is a small sample.
4. **Nonsense: the centered score separates it from typical careers, not from the weakest ones.** Nonsense queries reach at most about 0.06 centered (raw: up to 0.57, like real matches), below the median career's best match (about 0.12) but above the weakest careers' best (about 0.03). So a centered threshold alone could not tell a nonsense query from a career the collection barely covers. In the app the queries are always real careers and the LLM check decides, so this is a property of the score, not a risk to users.

**Limits:** the labels were drafted by an LLM (Claude) and accepted by you, so agreement with them is partly one model agreeing with another; only pooled candidates are judged, so a relevant claim that no score put in a top 5 is invisible here; and the ceiling (the careers with any relevant candidate, printed in the first table cell) is set by the sources, not by the retrieval.

The backend choice (exact NumPy over Chroma's approximate HNSW search) was measured separately: `scripts/analysis/compare_claim_search_backends.py`, CHANGELOG 030.